#**US_Sales_Analysis**

In [1]:
import os       #importing os to set environment variable
def install_java():
  !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null      #install openjdk
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"     #set environment variable
  !java -version       #check java version
install_java()

!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null


openjdk version "1.8.0_462"
OpenJDK Runtime Environment (build 1.8.0_462-8u462-ga~us1-0ubuntu2~22.04.2-b08)
OpenJDK 64-Bit Server VM (build 25.462-b08, mixed mode)
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 

In [2]:
pip install pyspark

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.appName("US_Sales_Analysis").getOrCreate()

In [5]:
sc = spark.sparkContext

In [6]:
from pyspark.sql.functions import *

In [7]:
from pyspark.sql.types import *

In [8]:
rawDataDF = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/content/sample_data/Data/US_Sales_Datasets.csv")

In [9]:
rawDataDF1 = rawDataDF.withColumn("Invoice Date", to_date(col("Invoice Date"), "dd-MM-yyyy"))

In [10]:
rawDataDF1.show(5)

+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|   Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|
+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|Foot Locker|    1185732|  2020-01-01|Northeast|New York|New York|Men's Street Foot...|            50|     1,200|   6,00,000|        3,00,000|             50%|    In-store|
|Foot Locker|    1185732|  2020-01-02|Northeast|New York|New York|Men's Athletic Fo...|            50|     1,000|   5,00,000|        1,50,000|             30%|    In-store|
|Foot Locker|    1185732|  2020-01-03|Northeast|New York|New York|Women's Street Fo...|            40|     1,000|   4,00,000|        1,

In [11]:
rawDataDF2 = rawDataDF1.withColumn("Gender", split(col("Product"), "'s ").getItem(0)).withColumn("Category", split(col("Product"), "'s ").getItem(1))

In [12]:
rawDataDF2.show(5)

+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|   Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|Gender|         Category|
+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|Foot Locker|    1185732|  2020-01-01|Northeast|New York|New York|Men's Street Foot...|            50|     1,200|   6,00,000|        3,00,000|             50%|    In-store|   Men|  Street Footwear|
|Foot Locker|    1185732|  2020-01-02|Northeast|New York|New York|Men's Athletic Fo...|            50|     1,000|   5,00,000|        1,50,000|             30%|    In-store|   Men|Athletic Footwear|
|Foot Lock

In [13]:
rawDataDF3 = rawDataDF2.withColumn("Units Sold", regexp_replace(col("Units Sold"),",","")).withColumn("Units Sold", col("Units Sold").cast("Integer"))

In [14]:
rawDataDF3.show(5)

+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|   Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|Gender|         Category|
+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|Foot Locker|    1185732|  2020-01-01|Northeast|New York|New York|Men's Street Foot...|            50|      1200|   6,00,000|        3,00,000|             50%|    In-store|   Men|  Street Footwear|
|Foot Locker|    1185732|  2020-01-02|Northeast|New York|New York|Men's Athletic Fo...|            50|      1000|   5,00,000|        1,50,000|             30%|    In-store|   Men|Athletic Footwear|
|Foot Lock

In [15]:
rawDataDF4 = rawDataDF3.withColumn("Operating Margin", regexp_replace(col("Operating Margin"),"%","")).withColumn("Operating Margin", col("Operating Margin").cast("Integer"))

In [16]:
rawDataDF4.show(5)

+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|   Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|Gender|         Category|
+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|Foot Locker|    1185732|  2020-01-01|Northeast|New York|New York|Men's Street Foot...|            50|      1200|   6,00,000|        3,00,000|              50|    In-store|   Men|  Street Footwear|
|Foot Locker|    1185732|  2020-01-02|Northeast|New York|New York|Men's Athletic Fo...|            50|      1000|   5,00,000|        1,50,000|              30|    In-store|   Men|Athletic Footwear|
|Foot Lock

In [17]:
rawDataDF5 = rawDataDF4.withColumn("Total Sales", col("Units Sold") * col("Price per Unit")).withColumn("Total Sales", col("Total Sales").cast("Double"))

In [18]:
rawDataDF5.show(5)

+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|   Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|Gender|         Category|
+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|Foot Locker|    1185732|  2020-01-01|Northeast|New York|New York|Men's Street Foot...|            50|      1200|    60000.0|        3,00,000|              50|    In-store|   Men|  Street Footwear|
|Foot Locker|    1185732|  2020-01-02|Northeast|New York|New York|Men's Athletic Fo...|            50|      1000|    50000.0|        1,50,000|              30|    In-store|   Men|Athletic Footwear|
|Foot Lock

In [19]:
rawDataDF6 = rawDataDF5.withColumn("Operating Profit", col("Total Sales") * col("Operating Margin")/100).withColumn("Operating Profit", col("Operating Profit").cast("Double"))

In [20]:
rawDataDF6.show(5)

+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|   Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|Gender|         Category|
+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|Foot Locker|    1185732|  2020-01-01|Northeast|New York|New York|Men's Street Foot...|            50|      1200|    60000.0|         30000.0|              50|    In-store|   Men|  Street Footwear|
|Foot Locker|    1185732|  2020-01-02|Northeast|New York|New York|Men's Athletic Fo...|            50|      1000|    50000.0|         15000.0|              30|    In-store|   Men|Athletic Footwear|
|Foot Lock

In [21]:
rawDataDF7 = rawDataDF6.withColumnRenamed("Invoice Date","Invoice_Date").withColumnRenamed("Total Sales","Total_Sales").withColumnRenamed("Operating Profit","Operating_Profit").withColumnRenamed("Units Sold","Units_Sold").withColumnRenamed("Sales Method", "Sales_Method")

In [22]:
rawDataDF7.show(5)

+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|   Retailer|Retailer ID|Invoice_Date|   Region|   State|    City|             Product|Price per Unit|Units_Sold|Total_Sales|Operating_Profit|Operating Margin|Sales_Method|Gender|         Category|
+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+------+-----------------+
|Foot Locker|    1185732|  2020-01-01|Northeast|New York|New York|Men's Street Foot...|            50|      1200|    60000.0|         30000.0|              50|    In-store|   Men|  Street Footwear|
|Foot Locker|    1185732|  2020-01-02|Northeast|New York|New York|Men's Athletic Fo...|            50|      1000|    50000.0|         15000.0|              30|    In-store|   Men|Athletic Footwear|
|Foot Lock

In [23]:
rawDataDF7.registerTempTable("sales")

/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [24]:
q1 = spark.sql("select year(Invoice_Date) as Year,round(sum(Total_Sales)/1000000, 2) as Total_Sales_in_Mill, round(sum(Operating_Profit)/1000000,2) as Total_Profit_in_Mill, round(sum(Total_Sales)/sum(Units_Sold),2) as Avg_Selling_Price, sum(Units_Sold) as Total_Units_Sold from sales group by year(Invoice_Date)")

In [25]:
q1.show()

+----+-------------------+--------------------+-----------------+----------------+
|Year|Total_Sales_in_Mill|Total_Profit_in_Mill|Avg_Selling_Price|Total_Units_Sold|
+----+-------------------+--------------------+-----------------+----------------+
|2020|              24.24|                9.02|            52.42|          462349|
|2021|              95.93|               38.21|            47.57|         2016512|
+----+-------------------+--------------------+-----------------+----------------+



In [26]:
q2 = spark.sql("select month(Invoice_Date) as Month, DATE_FORMAT(Invoice_Date, 'MMMM') AS MonthName, round(sum(Total_Sales)/1000000,2) as Total_Sales_in_Mill from sales group by month(Invoice_Date),DATE_FORMAT(Invoice_Date, 'MMMM') order by month(Invoice_Date)")

In [27]:
q2.show()

+-----+---------+-------------------+
|Month|MonthName|Total_Sales_in_Mill|
+-----+---------+-------------------+
|    1|  January|               9.74|
|    2| February|               8.26|
|    3|    March|               7.69|
|    4|    April|               9.69|
|    5|      May|              10.74|
|    6|     June|                9.8|
|    7|     July|              12.55|
|    8|   August|              12.29|
|    9|September|              10.41|
|   10|  October|               8.54|
|   11| November|               9.02|
|   12| December|              11.42|
+-----+---------+-------------------+



In [28]:
q3 = spark.sql("select state, round(sum(Total_Sales)/1000000,2) as Total_Sales_in_Mill from sales group by state order by state")

In [29]:
q3.show()

+-----------+-------------------+
|      state|Total_Sales_in_Mill|
+-----------+-------------------+
|    Alabama|               2.51|
|     Alaska|               1.81|
|    Arizona|               2.25|
|   Arkansas|                1.8|
| California|               8.58|
|   Colorado|               2.57|
|Connecticut|               1.65|
|   Delaware|               1.51|
|    Florida|               7.82|
|    Georgia|               2.71|
|     Hawaii|               2.73|
|      Idaho|               2.74|
|   Illinois|                1.2|
|    Indiana|               1.08|
|       Iowa|               0.91|
|     Kansas|               1.23|
|   Kentucky|               1.24|
|  Louisiana|               3.38|
|      Maine|               1.13|
|   Maryland|               0.95|
+-----------+-------------------+
only showing top 20 rows



In [30]:
q4 = spark.sql("select Sales_Method , round(sum(Total_Sales)/1000000,2) as Total_Sales_in_Mill from sales group by Sales_Method order by Sales_Method")

In [31]:
q4.show()

+------------+-------------------+
|Sales_Method|Total_Sales_in_Mill|
+------------+-------------------+
|    In-store|              35.66|
|      Online|              44.97|
|      Outlet|              39.54|
+------------+-------------------+



In [32]:
q5 = spark.sql("select Region, round(sum(Total_Sales)/1000000,2) as Total_Sales_in_Mill from sales group by Region order by Region")

In [33]:
q5.show()

+---------+-------------------+
|   Region|Total_Sales_in_Mill|
+---------+-------------------+
|  Midwest|              16.67|
|Northeast|              25.08|
|    South|               20.6|
|Southeast|              21.37|
|     West|              36.44|
+---------+-------------------+



In [34]:
q6 = spark.sql("select Category as Product, round(sum(Total_Sales)/1000000,2) as Total_Sales_in_Mill from sales group by Category")

In [35]:
q6.show()

+-----------------+-------------------+
|          Product|Total_Sales_in_Mill|
+-----------------+-------------------+
|          Apparel|              40.39|
|Athletic Footwear|              34.89|
|  Street Footwear|              44.88|
+-----------------+-------------------+



In [36]:
q7 = spark.sql("select Retailer, round(sum(Total_Sales)/1000000,2) as Total_Sales_in_Mill from sales group by Retailer order by Retailer")

In [37]:
q7.show()

+-------------+-------------------+
|     Retailer|Total_Sales_in_Mill|
+-------------+-------------------+
|       Amazon|               10.1|
|  Foot Locker|              29.02|
|       Kohl's|              13.51|
|Sports Direct|              24.62|
|      Walmart|              10.51|
|    West Gear|              32.41|
+-------------+-------------------+



In [38]:
q8a = spark.sql("select Category as Product_Category, sum(Units_Sold) as Total_Units from sales group by Category")

In [39]:
q8a.show()

+-----------------+-----------+
| Product_Category|Total_Units|
+-----------------+-----------+
|          Apparel|     740510|
|Athletic Footwear|     752762|
|  Street Footwear|     985589|
+-----------------+-----------+



In [40]:
q8b = spark.sql("select Gender as Gender_Type, sum(Units_Sold) as Total_Units from sales group by Gender")

In [41]:
q8b.show()

+-----------+-----------+
|Gender_Type|Total_Units|
+-----------+-----------+
|        Men|    1335529|
|      Women|    1143332|
+-----------+-----------+



In [42]:
q9 = spark.sql("select City , round(sum(Operating_Profit)/1000000,2) as Profit_in_Mill from sales group by City order by sum(Operating_Profit) desc")

In [43]:
q9.show()

+-------------+--------------+
|         City|Profit_in_Mill|
+-------------+--------------+
|     New York|          2.11|
|   Charleston|          2.02|
|San Francisco|          1.58|
|        Miami|          1.58|
|     Portland|          1.58|
|      Houston|          1.49|
|  New Orleans|          1.42|
|  Los Angeles|          1.38|
|   Birmingham|          1.37|
|      Orlando|          1.34|
|       Dallas|          1.34|
|    Knoxville|          1.27|
|    Charlotte|          1.26|
|        Boise|          1.22|
|       Albany|          1.22|
|     Richmond|          1.17|
|    Las Vegas|          1.08|
|      Detroit|          1.05|
|      Atlanta|          1.05|
|  Albuquerque|          1.03|
+-------------+--------------+
only showing top 20 rows

